<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-05/AI_Modul_5.7_Praktikum_Evaluasi_Model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 5.7: Praktikum Komputasi - Evaluasi Model dalam Machine Learning
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Tujuan Praktikum:
1. Menghitung dan membandingkan metrik evaluasi regresi (MAE, MSE, RMSE, MAPE, $R^2$, Adjusted $R^2$) secara manual dan terotomasi.
2. Membuktikan paradoks akurasi (*Accuracy Paradox*) pada dataset klasifikasi penyakit tanaman yang sangat tidak seimbang (*imbalanced*).
3. Mengonstruksi matriks konfusi dan menghitung metrik *Precision*, *Recall*, *Specificity*, serta $F_1$-Score.
4. Menggambar dan menafsirkan kurva ROC-AUC serta kurva *Precision-Recall*.
5. Menyetel ambang batas keputusan optimal (*Decision Threshold*) berbasis matriks biaya kerugian operasional agribisnis.

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, f1_score, fbeta_score
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150
print("Environment komputasi praktikum berhasil diinisialisasi.")


## 1. Evaluasi Kinerja Model Regresi: Estimasi Hasil Panen Sawit

Kita mengevaluasi model prediksi hasil panen Tandan Buah Segar (TBS) kelapa sawit ($ton/ha$) terhadap data uji aktual.


In [ ]:
np.random.seed(42)
n_val = 100

# Target Sejati Aktual (y_true) dan Prediksi Model AI (y_pred)
y_true = np.random.uniform(14.0, 32.0, n_val)
# Prediksi dengan sedikit bias dan deviasi acak
y_pred = y_true + np.random.normal(0.4, 1.8, n_val)
# Masukkan 2 sampel pencilan (misal sensor rusak)
y_pred[10] += 8.5
y_pred[50] -= 9.0

# 1. Perhitungan Manual Metrik Regresi
residu = y_true - y_pred
mae_manual = np.mean(np.abs(residu))
mse_manual = np.mean(residu**2)
rmse_manual = np.sqrt(mse_manual)
mape_manual = np.mean(np.abs(residu / y_true)) * 100.0

ss_res = np.sum(residu**2)
ss_tot = np.sum((y_true - np.mean(y_true))**2)
r2_manual = 1.0 - (ss_res / ss_tot)

d_fitur = 5 # Asumsi menggunakan 5 fitur prediktor
r2_adj = 1.0 - ((1.0 - r2_manual) * (n_val - 1) / (n_val - d_fitur - 1))

df_reg_metrics = pd.DataFrame({
    'Metrik Evaluasi': ['MAE (Ton/Ha)', 'MSE (Ton/Ha)^2', 'RMSE (Ton/Ha)', 'MAPE (%)', 'R2 Score', 'Adjusted R2'],
    'Nilai Terhitung': [mae_manual, mse_manual, rmse_manual, mape_manual, r2_manual, r2_adj]
})

print(df_reg_metrics.to_string(index=False))


## 2. Paradoks Akurasi pada Klasifikasi Penyakit Imbalanced

Mari kita simulasikan data deteksi dini infeksi jamur *Ganoderma* pada 2.000 pohon kelapa sawit, di mana hanya $5\%$ pohon yang terinfeksi (100 pohon sakit, 1.900 pohon sehat).
Kita bandingkan model naif (*Dummy Classifier*) yang selalu menebak "SEHAT" dengan *Random Forest*.


In [ ]:
X_imb, y_imb = make_classification(
    n_samples=2000,
    n_features=12,
    n_informative=6,
    weights=[0.95, 0.05],
    random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

# Model Naif (Dummy): Selalu menebak kelas mayoritas (Sehat = 0)
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_tr, y_tr)
y_pred_dummy = dummy_clf.predict(X_te)

# Model Machine Learning Nyata: Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_tr, y_tr)
y_pred_rf = rf_clf.predict(X_te)

print("=== AKURASI MODEL NAIF VS RANDOM FOREST ===")
print(f"Akurasi Model Naif (Tebak Sehat Terus) : {np.mean(y_pred_dummy == y_te)*100:.2f}% (Tampak Sempurna!)")
print(f"Akurasi Random Forest Classifier       : {np.mean(y_pred_rf == y_te)*100:.2f}%")

print("\n=== Matriks Konfusi Model Naif ===")
print(confusion_matrix(y_te, y_pred_dummy))
print("-> Seluruh 30 pohon sakit di data uji lolos (FN = 30, Recall = 0%)!")


## 3. Matriks Konfusi dan Metrik Turunan Lengkap

Visualisasi Matriks Konfusi Random Forest beserta perhitungan metrik *Precision*, *Recall*, *Specificity*, $F_1$, dan $F_2$-Score.


In [ ]:
cm_rf = confusion_matrix(y_te, y_pred_rf)
tn, fp, fn, tp = cm_rf.ravel()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = 2.0 * (precision * recall) / (precision + recall)
f2 = (1.0 + 2.0**2) * (precision * recall) / ((2.0**2 * precision) + recall)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Prediksi Sehat', 'Prediksi Sakit'],
            yticklabels=['Aktual Sehat', 'Aktual Sakit'])
plt.title('Matriks Konfusi Random Forest', fontweight='bold')
plt.ylabel('Kondisi Sejati Lapangan')
plt.xlabel('Prediksi Model AI')
plt.tight_layout()
plt.savefig('docs/assets/praktikum_confusion_matrix_rf.png', bbox_inches='tight')
plt.close()

print(f"True Positives (Pohon Sakit Terdeteksi) : {tp}")
print(f"False Positives (Alarm Palsu)           : {fp}")
print(f"False Negatives (Wabah Lolos Fatal)     : {fn}")
print(f"True Negatives (Pohon Sehat Terverifikasi): {tn}")
print("------------------------------------------")
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Specificity : {specificity:.4f}")
print(f"F1-Score    : {f1:.4f}")
print(f"F2-Score (Recall 2x Lebih Berbobot): {f2:.4f}")


## 4. Analisis Diskriminasi Kurva ROC-AUC dan Kurva Precision-Recall

Memvisualisasikan kinerja diskriminasi probabilitas Random Forest melintasi seluruh variasi ambang batas (*threshold*).


In [ ]:
y_probs = rf_clf.predict_proba(X_te)[:, 1]

fpr, tpr, _ = roc_curve(y_te, y_probs)
roc_auc = auc(fpr, tpr)

prec_vals, rec_vals, pr_thresh = precision_recall_curve(y_te, y_probs)
pr_auc = auc(rec_vals, prec_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Plot ROC
ax1.plot(fpr, tpr, color='#27ae60', linewidth=2.5, label=f'Random Forest (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], color='#7f8c8d', linestyle='--', label='Tebakan Acak (AUC = 0.500)')
ax1.set_title('Kurva ROC (Receiver Operating Characteristic)', fontweight='bold')
ax1.set_xlabel('False Positive Rate (FPR)')
ax1.set_ylabel('True Positive Rate (Recall)')
ax1.legend(loc='lower right')

# Plot Precision-Recall
ax2.plot(rec_vals, prec_vals, color='#2980b9', linewidth=2.5, label=f'Random Forest (PR-AUC = {pr_auc:.3f})')
ax2.set_title('Kurva Precision-Recall (Fokus Minoritas)', fontweight='bold')
ax2.set_xlabel('Recall (Sensitivitas)')
ax2.set_ylabel('Precision (Kemurnian)')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.savefig('docs/assets/praktikum_roc_pr_curves.png', bbox_inches='tight')
plt.close()
print(f"Evaluasi Kurva Selesai: ROC-AUC = {roc_auc:.4f}, PR-AUC = {pr_auc:.4f}")


## 5. Penyetelan Ambang Batas Berbasis Biaya Finansial Riil (*Cost Matrix*)

Dalam manajemen kebun kelapa sawit:
- Salah vonis pohon sehat (FP) memicu biaya cek mandor Rp 40.000,-
- Lolos pohon terinfeksi jamur (FN) memicu kerugian kematian pohon & penularan Rp 4.500.000,-
Kita mencari ambang batas probabilitas $\tau^*$ yang meminimalkan total kerugian operasional.


In [ ]:
c_fp = 40_000
c_fn = 4_500_000

thresholds_test = np.linspace(0.02, 0.98, 150)
biaya_operasional = []

for th in thresholds_test:
    p_pred = (y_probs >= th).astype(int)
    cm = confusion_matrix(y_te, p_pred)
    tn_i, fp_i, fn_i, tp_i = cm.ravel()
    cost = (fp_i * c_fp) + (fn_i * c_fn)
    biaya_operasional.append(cost)

biaya_operasional = np.array(biaya_operasional)
best_idx = np.argmin(biaya_operasional)
th_terbaik = thresholds_test[best_idx]
biaya_terendah = biaya_operasional[best_idx]

# Biaya pada threshold default 0.50
idx_default = np.argmin(np.abs(thresholds_test - 0.50))
biaya_default = biaya_operasional[idx_default]

print(f"Ambang Batas Standar (0.50)  -> Total Kerugian: Rp {biaya_default:,.0f}")
print(f"Ambang Batas Optimum ({th_terbaik:.3f}) -> Total Kerugian: Rp {biaya_terendah:,.0f}")
print(f"Penghematan Finansial Perkebunan: Rp {biaya_default - biaya_terendah:,.0f}!")


## 6. Latihan Terpandu Mandiri (Tugas Mahasiswa)

1. **Uji Ambang Youden's J**: Tentukan nilai ambang batas $\tau$ yang memaksimalkan indeks Youden $J = \text{TPR} - \text{FPR}$. Bandingkan ambang batas Youden ini dengan ambang batas berbasis biaya finansial!
2. **Evaluasi Multikelas**: Bangkitkan dataset klasifikasi tingkat kematangan kelapa sawit 4 kelas (`Mentah`, `Mengkal`, `Masak`, `Lewat Masak`). Hitung matriks konfusi $4 \times 4$ dan bandingkan metrik *Macro-Average F1* vs *Weighted-Average F1*.
3. **Audit Unsupervised Clustering**: Terapkan algoritma $K$-Means ($K=2$ hingga $K=6$) pada data zonasi kesuburan tanah. Hitung nilai *Silhouette Score* untuk tiap nilai $K$ dan tentukan klaster optimal.
